# 阶段5：Python读取与校验
按顺序理解：读取 → 筛选 → 关联 → 精确金额 → 分组 → 异常定位。小样本与SQL共同核对；此练习不写数据库。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src').is_dir()), None)
if root is None:
    raise RuntimeError('Open this notebook inside the project directory')
sys.path.insert(0, str(root / 'src'))
from etl.common import read_json
from etl.validation import parse_row, process_rows, Catalog


## 一行是什么
订单是一张订单，订单明细是一条商品成交。金额解析成Decimal；不会用缺失成本补0。


In [ ]:
orders = pd.DataFrame([parse_row('orders', r) for r in read_json(root / 'data/sample/stage3/orders.json')])
items = pd.DataFrame([parse_row('order_items', r) for r in read_json(root / 'data/sample/stage3/order_items.json')])
print('全部订单', len(orders), '全部明细', len(items))
orders.head()


## 筛选与关联
completed筛选对应SQL WHERE；merge对应JOIN。many_to_one检查订单一侧不会重复同一ID。


In [ ]:
completed = orders[orders['order_status'] == 'completed']
detail = items.merge(completed, on='order_id', how='inner', validate='many_to_one')
print('确认订单', len(completed), '确认明细', len(detail))
detail['sales_revenue'] = detail['quantity'] * detail['unit_list_price'] - detail['discount_amount']
detail['sales_cost'] = detail['quantity'] * detail['unit_cost_snapshot']
detail.groupby('product_id')[['sales_revenue', 'sales_cost']].sum()


练习：把分组字段改成channel_code，核对退款前收入C01=3300、C02=610、C03=240。这里还没减退款，所以不能与净收入3240直接比较。


In [ ]:
from etl.sources import load_facts
from etl.checks import verify_reference
facts = load_facts(root / 'configs/sample_files.json')
result = verify_reference(facts, root / 'tests/fixtures/sample_sql_reference.json')
print('所有基准一致:', result['passed'])
result['company']


## 坏日期不悄悄变成空值
一个有效日期和一个不存在的日期，程序接受前者并记录后者原因。


In [ ]:
rows = [{'customer_id':'LEARN_OK', 'signup_date':'2024-01-01'}, {'customer_id':'LEARN_BAD', 'signup_date':'2024-02-30'}]
accepted, rejected, skipped = process_rows('customers', rows, Catalog())
print('接受', len(accepted), '拒绝', len(rejected), '跳过', skipped)
rejected


下一步阅读validation.py里的退款累计检查和ingest.py里的事务。为何业务ID相同但内容不同不能直接覆盖？为何一个文件写入失败时批次仍应留痕？
